In [2]:
import pandas as pd
import duckdb

data = [
    # User A
    ["A", "2026-07-01 09:00:00", "O001", 100],
    ["A", "2026-07-01 10:30:00", "O002", 80],
    ["A", "2026-07-01 14:00:00", "O003", 120],

    # User B
    ["B", "2026-07-01 09:20:00", "O004", 60],
    ["B", "2026-07-01 11:00:00", "O005", 90],
    ["B", "2026-07-01 16:00:00", "O006", 150],

    # User C
    ["C", "2026-07-01 10:00:00", "O007", 200],
    ["C", "2026-07-01 15:30:00", "O008", 50],
]

df = pd.DataFrame(
    data,
    columns=["user_id", "order_time", "order_id", "amount"]
)

df["order_time"] = pd.to_datetime(df["order_time"])

print(df)



  user_id          order_time order_id  amount
0       A 2026-07-01 09:00:00     O001     100
1       A 2026-07-01 10:30:00     O002      80
2       A 2026-07-01 14:00:00     O003     120
3       B 2026-07-01 09:20:00     O004      60
4       B 2026-07-01 11:00:00     O005      90
5       B 2026-07-01 16:00:00     O006     150
6       C 2026-07-01 10:00:00     O007     200
7       C 2026-07-01 15:30:00     O008      50


# 题目要求

## 分别使用 SQL 和 Pandas 完成：

- 计算每个用户每次下单后的累计消费金额。

* **最终输出字段：**

    - `user_id`
    - `order_time`
    - `order_id`
    - `amount`
    - `running_total_amount`

In [ ]:
# SQL轨道

query = """
SELECT
    user_id,
    order_time,
    order_id,
    amount,
    SUM(amount) 
    OVER(PARTITION BY user_id ORDER BY order_time
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_total_amount
FROM df
ORDER BY user_id,order_time
"""
df_sql = duckdb.execute(query).fetchdf()
df_sql

,user_id,order_time,order_id,amount,running_total_amount
0,A,2026-07-01 09:00:00,O001,100,100.0
1,A,2026-07-01 10:30:00,O002,80,180.0
2,A,2026-07-01 14:00:00,O003,120,300.0
3,B,2026-07-01 09:20:00,O004,60,60.0
4,B,2026-07-01 11:00:00,O005,90,150.0
5,B,2026-07-01 16:00:00,O006,150,300.0
6,C,2026-07-01 10:00:00,O007,200,200.0
7,C,2026-07-01 15:30:00,O008,50,250.0


In [10]:
# PANDAS轨道

df_pd = (
    df
    .sort_values(by=['user_id','order_time'],ascending=[True,True])
    .assign(
        running_total_amount = lambda x:(
            x.groupby('user_id')['amount']
            .cumsum()
            
        )
    )
    .reset_index(drop=True)
)
df_pd

,user_id,order_time,order_id,amount,running_total_amount
0,A,2026-07-01 09:00:00,O001,100,100
1,A,2026-07-01 10:30:00,O002,80,180
2,A,2026-07-01 14:00:00,O003,120,300
3,B,2026-07-01 09:20:00,O004,60,60
4,B,2026-07-01 11:00:00,O005,90,150
5,B,2026-07-01 16:00:00,O006,150,300
6,C,2026-07-01 10:00:00,O007,200,200
7,C,2026-07-01 15:30:00,O008,50,250
